# nb_silver_stream_vehicle_battery

**Source :** Bronze Delta stream — `event-stream/vehicle_battery_live/`  
**Sink   :** `abfss://silver@evdatalakedev2.dfs.core.windows.net/sl_vehicle_battery_live/`  
**Quarantine:** `abfss://silver@evdatalakedev2.dfs.core.windows.net/quarantine/vehicle_battery_invalid/`  
**Trigger:** Every 60 seconds via `foreachBatch`

Uses `foreachBatch` instead of plain `writeStream` because:  
- MERGE (upsert on `event_id`) is not supported in streaming output mode  
- We need to split clean vs bad rows per batch  
- We apply 9 named DQ rules with per-row explanations

**DQ Rules applied per row:**

| Rule | Condition |
|---|---|
| R01 | event_id IS NOT NULL |
| R02 | vehicle_id IS NOT NULL |
| R03 | session_id IS NOT NULL |
| R04 | battery_pct BETWEEN 0 AND 100 |
| R05 | charging_rate_kw BETWEEN 0 AND 350 |
| R06 | battery_temp_c BETWEEN -10 AND 80 |
| R07 | state_of_charge_target_pct BETWEEN 1 AND 100 |
| R08 | event_ts IS NOT NULL |
| R09 | _is_corrupt == False |

In [ ]:
# ── Cell 1: secrets ───────────────────────────────────────────────────────────
SP_CLIENT_ID  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-id")
SP_CLIENT_SEC = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-secret")
SP_TENANT_ID  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-tenant-id")
ADLS_ACCOUNT  = dbutils.secrets.get(scope="ev-kv-scope", key="adls-account-name")

In [ ]:
# ── Cell 2: ADLS OAuth config ─────────────────────────────────────────────────
spark.conf.set(f"fs.azure.account.auth.type.{ADLS_ACCOUNT}.dfs.core.windows.net",       "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{ADLS_ACCOUNT}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{ADLS_ACCOUNT}.dfs.core.windows.net",     SP_CLIENT_ID)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{ADLS_ACCOUNT}.dfs.core.windows.net", SP_CLIENT_SEC)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{ADLS_ACCOUNT}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token")
print("ADLS OAuth configured.")

In [ ]:
# ── Cell 3: paths ─────────────────────────────────────────────────────────────
BRONZE_PATH     = f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/event-stream/vehicle_battery_live/"
SILVER_PATH     = f"abfss://silver@{ADLS_ACCOUNT}.dfs.core.windows.net/sl_vehicle_battery_live/"
QUARANTINE_PATH = f"abfss://silver@{ADLS_ACCOUNT}.dfs.core.windows.net/quarantine/vehicle_battery_invalid/"
CHECKPOINT_PATH = f"abfss://silver@{ADLS_ACCOUNT}.dfs.core.windows.net/_checkpoints/vehicle-battery-live/"

print(f"Source (Bronze) : {BRONZE_PATH}")
print(f"Sink   (Silver) : {SILVER_PATH}")
print(f"Quarantine      : {QUARANTINE_PATH}")

In [ ]:
# ── Cell 4: DQ + transform function ──────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable
import time

def transform_to_silver(batch_df, batch_id):
    print(f"\n[Batch {batch_id}] input rows = {batch_df.count()}")
    if batch_df.rdd.isEmpty():
        print(f"[Batch {batch_id}] empty — skipping.")
        return

    # ── Apply DQ rules ────────────────────────────────────────────────────────
    tagged = (
        batch_df
        .withColumn("_r01", F.col("event_id").isNotNull())
        .withColumn("_r02", F.col("vehicle_id").isNotNull())
        .withColumn("_r03", F.col("session_id").isNotNull())
        .withColumn("_r04", F.col("battery_pct").between(0, 100))
        .withColumn("_r05", F.col("charging_rate_kw").between(0, 350))
        .withColumn("_r06", F.col("battery_temp_c").between(-10, 80))
        .withColumn("_r07", F.col("state_of_charge_target_pct").between(1, 100))
        .withColumn("_r08", F.col("event_ts").isNotNull())
        .withColumn("_r09", F.col("_is_corrupt") == False)
    )

    # Build human-readable reason string for failed rows
    RULE_LABELS = {
        "_r01": "event_id null",
        "_r02": "vehicle_id null",
        "_r03": "session_id null",
        "_r04": "battery_pct out of range",
        "_r05": "charging_rate_kw out of range",
        "_r06": "battery_temp_c out of range",
        "_r07": "soc_target out of range",
        "_r08": "event_ts null",
        "_r09": "corrupt JSON body",
    }
    reason_expr = F.concat_ws(", ", *[
        F.when(~F.col(r), F.lit(label)) for r, label in RULE_LABELS.items()
    ])
    tagged = (
        tagged
        .withColumn("_dq_reasons", reason_expr)
        .withColumn("_dq_passed",  F.col("_dq_reasons") == "")
    )

    # ── Split clean vs bad ────────────────────────────────────────────────────
    clean_df = tagged.filter(F.col("_dq_passed"))
    bad_df   = tagged.filter(~F.col("_dq_passed"))

    bad_count   = bad_df.count()
    clean_count = clean_df.count()
    print(f"[Batch {batch_id}] clean={clean_count}  quarantine={bad_count}")

    # ── Deduplicate within batch on event_id (keep latest _ingestion_ts) ──────
    w = Window.partitionBy("event_id").orderBy(F.col("_ingestion_ts").desc())
    deduped = (
        clean_df
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn", "_r01", "_r02", "_r03", "_r04", "_r05", "_r06", "_r07", "_r08", "_r09",
               "_dq_reasons", "_dq_passed")
        .withColumn("sl_ingested_at", F.current_timestamp())
        .withColumn("sl_batch_id",    F.lit(str(batch_id)))
    )

    # ── MERGE into Silver Delta ───────────────────────────────────────────────
    if DeltaTable.isDeltaTable(spark, SILVER_PATH):
        tgt = DeltaTable.forPath(spark, SILVER_PATH)
        (
            tgt.alias("t")
               .merge(deduped.alias("s"), "t.event_id = s.event_id")
               .whenMatchedUpdateAll()
               .whenNotMatchedInsertAll()
               .execute()
        )
    else:
        # First run — create the table
        deduped.write.format("delta").mode("overwrite").save(SILVER_PATH)

    print(f"[Batch {batch_id}] Silver MERGE done.")

    # ── Quarantine bad rows ───────────────────────────────────────────────────
    if bad_count > 0:
        quarantine_df = (
            bad_df
            .withColumn("_quarantine_ts", F.current_timestamp())
            .withColumn("_batch_id",      F.lit(str(batch_id)))
            .withColumn("_source",        F.lit("eventhub/vehicle-battery-live"))
            .drop("_r01", "_r02", "_r03", "_r04", "_r05", "_r06", "_r07", "_r08", "_r09", "_dq_passed")
        )
        quarantine_df.write.format("delta").mode("append").save(QUARANTINE_PATH)
        print(f"[Batch {batch_id}] {bad_count} rows quarantined.")

In [ ]:
# ── Cell 5: read Bronze stream + trigger foreachBatch ─────────────────────────
bronze_stream = (
    spark.readStream
    .format("delta")
    .load(BRONZE_PATH)
)

silver_query = (
    bronze_stream
    .writeStream
    .foreachBatch(transform_to_silver)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="60 seconds")
    .start()
)

print(f"Silver stream started. Query ID: {silver_query.id}")

In [ ]:
# ── Cell 6: keep-alive monitor loop ──────────────────────────────────────────
import time

while silver_query.isActive:
    prog = silver_query.lastProgress
    if prog:
        print(f"[Silver] batch={prog.get('batchId','?')}  inputRows={prog.get('numInputRows',0)}")
    time.sleep(60)

silver_query.awaitTermination()

In [ ]:
# ── Cell 7: optional verify (run manually) ────────────────────────────────────
silver_df = spark.read.format("delta").load(SILVER_PATH)
print(f"Total Silver rows: {silver_df.count()}")
silver_df.orderBy(F.col("sl_ingested_at").desc()).limit(5).display()

# Check quarantine
try:
    q_df = spark.read.format("delta").load(QUARANTINE_PATH)
    print(f"Quarantine rows : {q_df.count()}")
    q_df.select("event_id", "_dq_reasons", "_quarantine_ts").limit(5).display()
except Exception:
    print("No quarantine records yet.")

In [ ]:
# ── Cell 8: DQ summary per batch (run manually) ───────────────────────────────
spark.sql(f"""
    SELECT
        sl_batch_id,
        COUNT(*)                           AS total_rows,
        COUNT(DISTINCT vehicle_id)         AS unique_vehicles,
        ROUND(AVG(battery_pct), 2)         AS avg_battery_pct,
        ROUND(AVG(charging_rate_kw), 2)    AS avg_rate_kw,
        ROUND(AVG(battery_temp_c), 2)      AS avg_temp_c,
        MIN(sl_ingested_at)                AS batch_start,
        MAX(sl_ingested_at)                AS batch_end
    FROM delta.`{SILVER_PATH}`
    GROUP BY sl_batch_id
    ORDER BY sl_batch_id DESC
""").display()